# Inventory Optimization — Multivariate Time-Series Forecasting

## Objective

This stage extends the inventory management framework into an inventory optimization simulation.

The objective is to determine replenishment quantities and evaluate inventory performance under a defined replenishment policy.

The analysis will evaluate:

- Order Quantity
- Reorder Frequency
- Stockout Days
- Average Inventory
- Service Level
- Inventory Turnover
- Holding Cost
- Ordering Cost

## Inventory Optimization Flow

```text
Demand Information
        ↓
Reorder Point
        ↓
Order Quantity
        ↓
Replenishment Policy
        ↓
Inventory Simulation
        ↓
Inventory Performance

In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
import pyarrow.parquet as pq

DATA_PATH = "/content/drive/MyDrive/features_event_snap.parquet"

TRAIN_END_DATE = pd.Timestamp("2016-03-27")

LEAD_TIME_DAYS = 7
SERVICE_LEVEL_Z = 1.645

print("Configuration set successfully.")

Configuration set successfully.


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


##Historical Demand Statistics

In [4]:
demand_stats = []

parquet_file = pq.ParquetFile(DATA_PATH)

for rg_idx in range(parquet_file.num_row_groups):

    table = parquet_file.read_row_group(
        rg_idx,
        columns=["item_id", "store_id", "date", "sales"]
    )

    df = table.to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    df = df[df["date"] <= TRAIN_END_DATE]

    if len(df) > 0:
        grouped = (
            df.groupby(["item_id", "store_id"])
            .agg(
                total_demand=("sales", "sum"),
                total_days=("sales", "count")
            )
            .reset_index()
        )

        demand_stats.append(grouped)

    del table
    del df

    if (rg_idx + 1) % 10 == 0:
        print(f"Processed {rg_idx + 1}/{parquet_file.num_row_groups} row groups")

print("All row groups processed.")

Processed 10/61 row groups
Processed 20/61 row groups
Processed 30/61 row groups
Processed 40/61 row groups
Processed 50/61 row groups
Processed 60/61 row groups
All row groups processed.


##Aggregate Demand Statistics

In [5]:
demand_stats_df = pd.concat(
    demand_stats,
    ignore_index=True
)

demand_stats_df = (
    demand_stats_df
    .groupby(["item_id", "store_id"], as_index=False)
    .agg(
        total_demand=("total_demand", "sum"),
        total_days=("total_days", "sum")
    )
)

demand_stats_df["average_daily_demand"] = (
    demand_stats_df["total_demand"] /
    demand_stats_df["total_days"]
)

print("Demand statistics shape:", demand_stats_df.shape)
print("Unique item-store series:",
      demand_stats_df[["item_id", "store_id"]].drop_duplicates().shape[0])

print("\nTotal days per series:")
print(demand_stats_df["total_days"].value_counts().head())

print("\nAverage daily demand summary:")
print(demand_stats_df["average_daily_demand"].describe())

Demand statistics shape: (30490, 5)
Unique item-store series: 30490

Total days per series:
total_days
1885    30490
Name: count, dtype: int64

Average daily demand summary:
count    30490.000000
mean         1.122458
std          2.729786
min          0.000531
25%          0.183024
50%          0.438727
75%          1.056101
max        131.253050
Name: average_daily_demand, dtype: float64


##Forecast Error Statistics

In [6]:
FORECAST_PATH = "/content/drive/MyDrive/transformer_validation_forecasts.parquet"

forecast_df = pd.read_parquet(FORECAST_PATH)

forecast_df["forecast_error"] = (
    forecast_df["actual_demand"] -
    forecast_df["forecast_demand"]
)

error_stats = (
    forecast_df
    .groupby(["item_id", "store_id"])
    .agg(
        mean_forecast_error=("forecast_error", "mean"),
        std_forecast_error=("forecast_error", "std")
    )
    .reset_index()
)

print("Forecast data shape:", forecast_df.shape)
print("Error statistics shape:", error_stats.shape)
print("\nMissing values:")
print(error_stats.isna().sum())

Forecast data shape: (853720, 6)
Error statistics shape: (30490, 4)

Missing values:
item_id                0
store_id               0
mean_forecast_error    0
std_forecast_error     0
dtype: int64


##Safety Stock Calculate

In [7]:
error_stats["safety_stock"] = (
    SERVICE_LEVEL_Z
    * error_stats["std_forecast_error"]
    * np.sqrt(LEAD_TIME_DAYS)
)

print("Safety stock summary:")
print(error_stats["safety_stock"].describe())

Safety stock summary:
count    30490.000000
mean         5.480898
std          6.760612
min          0.000000
25%          2.255288
50%          3.785751
75%          6.307819
max        173.753326
Name: safety_stock, dtype: float64


##Combine Demand and Safety Stock

In [8]:
inventory_policy = demand_stats_df.merge(
    error_stats,
    on=["item_id", "store_id"],
    how="inner"
)

inventory_policy["lead_time_demand"] = (
    inventory_policy["average_daily_demand"]
    * LEAD_TIME_DAYS
)

inventory_policy["reorder_point"] = (
    inventory_policy["lead_time_demand"]
    + inventory_policy["safety_stock"]
)

print("Inventory policy shape:", inventory_policy.shape)

print("\nMissing values:")
print(
    inventory_policy[
        ["average_daily_demand",
         "safety_stock",
         "lead_time_demand",
         "reorder_point"]
    ].isna().sum()
)

Inventory policy shape: (30490, 10)

Missing values:
average_daily_demand    0
safety_stock            0
lead_time_demand        0
reorder_point           0
dtype: int64


In [9]:
ORDERING_COST = 100      # assumed cost per order
HOLDING_COST = 20        # assumed annual holding cost per unit
DAYS_PER_YEAR = 365

inventory_policy["annual_demand"] = (
    inventory_policy["average_daily_demand"]
    * DAYS_PER_YEAR
)

inventory_policy["order_quantity"] = np.sqrt(
    (2 * inventory_policy["annual_demand"] * ORDERING_COST)
    / HOLDING_COST
)

print("Order quantity summary:")
print(inventory_policy["order_quantity"].describe())

Order quantity summary:
count    30490.000000
mean        50.587994
std         39.215794
min          1.391524
25%         25.846414
50%         40.016906
75%         62.086777
max        692.151453
Name: order_quantity, dtype: float64


##Inventory Simulation

In [10]:
inventory_policy["initial_inventory"] = (
    2 * inventory_policy["reorder_point"]
)

print("Initial inventory summary:")
print(inventory_policy["initial_inventory"].describe())

Initial inventory summary:
count    30490.000000
mean        26.676215
std         48.880301
min          0.089125
25%          7.568980
50%         14.166027
75%         27.372983
max       2140.378307
Name: initial_inventory, dtype: float64


In [11]:
simulation_df = forecast_df[
    ["item_id", "store_id", "date", "actual_demand"]
].copy()

simulation_df = simulation_df.merge(
    inventory_policy[
        [
            "item_id",
            "store_id",
            "reorder_point",
            "order_quantity",
            "initial_inventory"
        ]
    ],
    on=["item_id", "store_id"],
    how="left"
)

simulation_df = simulation_df.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

print("Simulation data shape:", simulation_df.shape)

print("\nMissing values:")
print(
    simulation_df[
        [
            "reorder_point",
            "order_quantity",
            "initial_inventory"
        ]
    ].isna().sum()
)

Simulation data shape: (853720, 7)

Missing values:
reorder_point        0
order_quantity       0
initial_inventory    0
dtype: int64


##Replenishment Simulation

In [12]:
LEAD_TIME_DAYS = 7

# Fast lookup for inventory policy
policy_lookup = (
    inventory_policy
    .set_index(["item_id", "store_id"])[
        ["reorder_point", "order_quantity", "initial_inventory"]
    ]
    .to_dict("index")
)

simulation_results = []

for (item_id, store_id), group in simulation_df.groupby(
    ["item_id", "store_id"],
    sort=False
):

    group = group.sort_values("date").reset_index(drop=True)

    policy = policy_lookup[(item_id, store_id)]

    inventory = policy["initial_inventory"]
    reorder_point = policy["reorder_point"]
    order_quantity = policy["order_quantity"]

    # Orders waiting for arrival
    pending_orders = {}

    for i, row in group.iterrows():

        current_date = row["date"]

        # Receive orders whose lead time has finished
        received_quantity = pending_orders.pop(
            current_date,
            0
        )

        inventory += received_quantity

        # Daily demand
        demand = row["actual_demand"]

        inventory_before_demand = inventory

        # Fulfill demand
        inventory = max(
            inventory - demand,
            0
        )

        stockout = demand > inventory_before_demand

        # Calculate quantity already on order
        on_order = sum(pending_orders.values())

        # Inventory position
        inventory_position = inventory + on_order

        # Reorder decision
        reorder_signal = inventory_position <= reorder_point

        order_placed = False
        order_arrival_date = pd.NaT

        if reorder_signal:

            order_arrival_date = (
                current_date
                + pd.Timedelta(days=LEAD_TIME_DAYS)
            )

            pending_orders[order_arrival_date] = (
                pending_orders.get(order_arrival_date, 0)
                + order_quantity
            )

            order_placed = True

        simulation_results.append({
            "item_id": item_id,
            "store_id": store_id,
            "date": current_date,
            "actual_demand": demand,
            "inventory_before_demand": inventory_before_demand,
            "inventory_remaining": inventory,
            "on_order": on_order,
            "inventory_position": inventory_position,
            "reorder_point": reorder_point,
            "order_quantity": order_quantity,
            "reorder_signal": reorder_signal,
            "order_placed": order_placed,
            "order_arrival_date": order_arrival_date,
            "stockout": stockout
        })

print("Simulation completed.")

Simulation completed.


##Simulation Output Validation

In [13]:
simulation_results_df = pd.DataFrame(simulation_results)

print("Simulation shape:", simulation_results_df.shape)

print("\nColumns:")
print(simulation_results_df.columns.tolist())

print("\nMissing values:")
print(simulation_results_df.isna().sum())

print("\nReorder orders placed:")
print(simulation_results_df["order_placed"].value_counts())

print("\nStockout observations:")
print(simulation_results_df["stockout"].value_counts())

Simulation shape: (853720, 14)

Columns:
['item_id', 'store_id', 'date', 'actual_demand', 'inventory_before_demand', 'inventory_remaining', 'on_order', 'inventory_position', 'reorder_point', 'order_quantity', 'reorder_signal', 'order_placed', 'order_arrival_date', 'stockout']

Missing values:
item_id                         0
store_id                        0
date                            0
actual_demand                   0
inventory_before_demand         0
inventory_remaining             0
on_order                        0
inventory_position              0
reorder_point                   0
order_quantity                  0
reorder_signal                  0
order_placed                    0
order_arrival_date         824577
stockout                        0
dtype: int64

Reorder orders placed:
order_placed
False    824577
True      29143
Name: count, dtype: int64

Stockout observations:
stockout
False    840946
True      12774
Name: count, dtype: int64


In [14]:
sample_series = simulation_results_df[
    (simulation_results_df["item_id"] == "HOBBIES_1_008") &
    (simulation_results_df["store_id"] == "CA_1")
].copy()

print(sample_series[
    [
        "date",
        "actual_demand",
        "inventory_before_demand",
        "inventory_remaining",
        "on_order",
        "inventory_position",
        "reorder_point",
        "order_quantity",
        "order_placed",
        "order_arrival_date",
        "stockout"
    ]
].to_string(index=False))

      date  actual_demand  inventory_before_demand  inventory_remaining   on_order  inventory_position  reorder_point  order_quantity  order_placed order_arrival_date  stockout
2016-03-28            0.0               191.598007           191.598007   0.000000          191.598007      95.799003      162.290142         False                NaT     False
2016-03-29           11.0               191.598007           180.598007   0.000000          180.598007      95.799003      162.290142         False                NaT     False
2016-03-30            5.0               180.598007           175.598007   0.000000          175.598007      95.799003      162.290142         False                NaT     False
2016-03-31            2.0               175.598007           173.598007   0.000000          173.598007      95.799003      162.290142         False                NaT     False
2016-04-01           12.0               173.598007           161.598007   0.000000          161.598007      95.7990

##Inventory Performance Metrics

In [15]:
total_orders = simulation_results_df["order_placed"].sum()
total_stockout_days = simulation_results_df["stockout"].sum()

service_level = (
    1 - total_stockout_days / len(simulation_results_df)
) * 100

average_inventory = (
    simulation_results_df["inventory_remaining"].mean()
)

total_demand = simulation_results_df["actual_demand"].sum()

total_ordered_quantity = (
    simulation_results_df.loc[
        simulation_results_df["order_placed"],
        "order_quantity"
    ].sum()
)

inventory_turnover = (
    total_demand / average_inventory
)

print("Inventory Performance")
print("-" * 40)
print(f"Total demand: {total_demand:,.2f}")
print(f"Total orders placed: {total_orders:,}")
print(f"Total quantity ordered: {total_ordered_quantity:,.2f}")
print(f"Average inventory: {average_inventory:,.2f}")
print(f"Stockout days: {total_stockout_days:,}")
print(f"Service level: {service_level:.2f}%")
print(f"Inventory turnover: {inventory_turnover:.2f}")

Inventory Performance
----------------------------------------
Total demand: 1,183,626.00
Total orders placed: 29,143
Total quantity ordered: 1,621,645.21
Average inventory: 24.34
Stockout days: 12,774
Service level: 98.50%
Inventory turnover: 48626.20


##Inventory Cost Analysis

In [16]:
VALIDATION_DAYS = 28

total_ordering_cost = (
    total_orders * ORDERING_COST
)

total_holding_cost = (
    average_inventory
    * HOLDING_COST
    * VALIDATION_DAYS
    / DAYS_PER_YEAR
)

total_inventory_cost = (
    total_ordering_cost
    + total_holding_cost
)

print("Inventory Cost Analysis")
print("-" * 40)
print(f"Ordering cost: ₹{total_ordering_cost:,.2f}")
print(f"Holding cost: ₹{total_holding_cost:,.2f}")
print(f"Total inventory cost: ₹{total_inventory_cost:,.2f}")

Inventory Cost Analysis
----------------------------------------
Ordering cost: ₹2,914,300.00
Holding cost: ₹37.35
Total inventory cost: ₹2,914,337.35


##Correct Cost & Turnover Metrics

In [17]:
NUMBER_OF_SERIES = inventory_policy.shape[0]

total_average_inventory = (
    average_inventory * NUMBER_OF_SERIES
)

correct_holding_cost = (
    total_average_inventory
    * HOLDING_COST
    * VALIDATION_DAYS
    / DAYS_PER_YEAR
)

correct_total_inventory_cost = (
    total_ordering_cost
    + correct_holding_cost
)

# Average inventory per item-store series
average_inventory_per_series = (
    total_average_inventory / NUMBER_OF_SERIES
)

# Turnover over the 28-day validation period
inventory_turnover_28d = (
    total_demand / total_average_inventory
)

# Annualized turnover
annualized_turnover = (
    inventory_turnover_28d
    * DAYS_PER_YEAR
    / VALIDATION_DAYS
)

print("Corrected Inventory Metrics")
print("-" * 45)
print(f"Number of item-store series: {NUMBER_OF_SERIES:,}")
print(f"Total average inventory: {total_average_inventory:,.2f}")
print(f"Average inventory per series: {average_inventory_per_series:.2f}")
print(f"Ordering cost: ₹{total_ordering_cost:,.2f}")
print(f"Holding cost: ₹{correct_holding_cost:,.2f}")
print(f"Total inventory cost: ₹{correct_total_inventory_cost:,.2f}")
print(f"28-day inventory turnover: {inventory_turnover_28d:.2f}")
print(f"Annualized inventory turnover: {annualized_turnover:.2f}")

Corrected Inventory Metrics
---------------------------------------------
Number of item-store series: 30,490
Total average inventory: 742,166.98
Average inventory per series: 24.34
Ordering cost: ₹2,914,300.00
Holding cost: ₹1,138,667.15
Total inventory cost: ₹4,052,967.15
28-day inventory turnover: 1.59
Annualized inventory turnover: 20.79


##Final Validation

In [18]:
print("Final Simulation Validation")
print("-" * 45)

print(
    "Negative inventory:",
    (simulation_results_df["inventory_remaining"] < 0).sum()
)

print(
    "Non-positive order quantity:",
    (simulation_results_df["order_quantity"] <= 0).sum()
)

print(
    "Non-positive reorder point:",
    (simulation_results_df["reorder_point"] <= 0).sum()
)

print(
    "Duplicate item-store-date rows:",
    simulation_results_df.duplicated(
        ["item_id", "store_id", "date"]
    ).sum()
)

print(
    "Orders placed:",
    simulation_results_df["order_placed"].sum()
)

print(
    "Orders with arrival date:",
    simulation_results_df.loc[
        simulation_results_df["order_placed"],
        "order_arrival_date"
    ].notna().sum()
)

Final Simulation Validation
---------------------------------------------
Negative inventory: 0
Non-positive order quantity: 0
Non-positive reorder point: 0
Duplicate item-store-date rows: 0
Orders placed: 29143
Orders with arrival date: 29143


# Phase 12 — Inventory Optimization

## Objective

This stage extends the inventory management framework into an inventory optimization simulation.

The objective is to determine replenishment quantities and evaluate inventory performance under a defined replenishment policy.

The analysis uses the item-store level reorder point calculated in the previous inventory management stage and introduces an EOQ-based order quantity.

---

## Data Used

- Historical training demand: 2011-01-29 to 2016-03-27
- Validation demand: 2016-03-28 to 2016-04-24
- Item-store series: 30,490
- Validation observations: 853,720

---

## Inventory Assumptions

| Parameter | Value |
|---|---:|
| Lead Time | 7 days |
| Service Level | 95% |
| Z-value | 1.645 |
| Ordering Cost | ₹100 per order |
| Holding Cost | ₹20 per unit per year |
| Initial Inventory | 2 × Reorder Point |

The ordering and holding costs are scenario assumptions because the M5 dataset does not provide actual inventory-related cost information.

---

## Order Quantity

The Economic Order Quantity (EOQ) formulation was used:

\[
EOQ = \sqrt{\frac{2DS}{H}}
\]

where:

- \(D\) = annual demand
- \(S\) = ordering cost per order
- \(H\) = annual holding cost per unit

The resulting order quantity statistics were:

- Mean: 50.59 units
- Median: 40.02 units
- Minimum: 1.39 units
- Maximum: 692.15 units

---

## Replenishment Policy

A replenishment order is triggered when the inventory position reaches or falls below the reorder point.

Inventory position is defined as:

\[
Inventory\ Position =
On\text{-}Hand\ Inventory + On\text{-}Order\ Inventory
\]

Each replenishment order uses the calculated EOQ quantity.

Orders are received after the assumed 7-day lead time rather than immediately.

---

## Inventory Simulation

The replenishment policy was simulated over the 28-day validation period.

The simulation tracks:

- Daily demand
- Inventory before demand
- Remaining inventory
- Inventory on order
- Inventory position
- Reorder point
- Order quantity
- Reorder signals
- Order placement
- Expected order arrival
- Stockout observations

---

## Simulation Results

| Metric | Result |
|---|---:|
| Item-store series | 30,490 |
| Total demand | 1,183,626 units |
| Orders placed | 29,143 |
| Total quantity ordered | 1,621,645.21 units |
| Average inventory per series | 24.34 units |
| Total average inventory | 742,166.98 units |
| Stockout observations | 12,774 |
| Simulated service level | 98.50% |
| 28-day inventory turnover | 1.59× |
| Annualized inventory turnover | 20.79× |
| Ordering cost | ₹2,914,300 |
| Holding cost | ₹1,138,667.15 |
| Total inventory cost | ₹4,052,967.15 |

---

## Service Level

The simulated service level was calculated as:

\[
Service\ Level =
\left(
1 -
\frac{Stockout\ Observations}
{Total\ Observations}
\right)
\times 100
\]

The resulting observation-level service measure was **98.50%**.

This should not be interpreted as an actual historical business fill rate because the dataset does not contain real inventory or replenishment records.

---

## Inventory Turnover

The 28-day turnover was calculated using:

\[
Inventory\ Turnover =
\frac{Total\ Validation\ Demand}
{Total\ Average\ Inventory}
\]

The resulting turnover was **1.59×** over the 28-day validation period.

Annualizing this measure using the 365/28 scaling factor resulted in an annualized scenario turnover of **20.79×**.

---

## Cost Analysis

Ordering cost was calculated as:

\[
Ordering\ Cost =
Number\ of\ Orders \times Ordering\ Cost\ per\ Order
\]

Holding cost was calculated over the 28-day validation period:

\[
Holding\ Cost =
Average\ Inventory
\times Annual\ Holding\ Cost
\times \frac{28}{365}
\]

The resulting scenario costs were:

- Ordering Cost: ₹2,914,300
- Holding Cost: ₹1,138,667.15
- Total Inventory Cost: ₹4,052,967.15

These costs are simulated values based on assumed cost parameters.

---

## Validation Checks

The final simulation passed the following checks:

- Negative inventory: 0
- Non-positive order quantity: 0
- Non-positive reorder point: 0
- Duplicate item-store-date rows: 0
- Orders placed: 29,143
- Orders with arrival dates: 29,143

The replenishment simulation therefore produced structurally consistent results under the defined assumptions.

---

## Limitations

1. The dataset does not contain actual on-hand inventory.
2. Supplier lead time is assumed to be 7 days.
3. Ordering cost is assumed to be ₹100 per order.
4. Holding cost is assumed to be ₹20 per unit per year.
5. Initial inventory is assumed to be twice the reorder point.
6. Actual supplier replenishment quantities are not available.
7. The service-level measure is based on simulation observations and is not an actual historical fill rate.
8. Inventory turnover is calculated from the simulated inventory scenario.
9. The safety-stock calculation assumes forecast-error variability and uses a simplified lead-time formulation.
10. The simulation uses historical validation demand rather than actual historical inventory transactions.

---

## Conclusion

This stage extends the demand forecasting pipeline into an inventory optimization framework.

An EOQ-based replenishment quantity was calculated for each item-store series, and a 7-day lead-time replenishment policy was simulated over the validation period.

The simulation provides a structured framework for evaluating replenishment frequency, inventory levels, stockouts, service level, turnover, and inventory-related costs.

Because actual inventory and cost data are unavailable, the results represent a scenario-based inventory optimization analysis rather than real historical inventory performance.

**Phase 12 — Inventory Optimization: COMPLETE**